# 1. Setup: imports \& drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from __future__ import annotations

import argparse
import asyncio
import json
import os
import time
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from openai import AsyncOpenAI, RateLimitError


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. OpenAI setup and configuration

In [2]:


@dataclass
class RuntimeConfig:
    model: str = "gpt-4o-mini"
    batch_size: int = 20
    max_concurrent: int = 15
    max_retries: int = 3
    sleep_sec: float = 2.0
    checkpoint_every_sec: int = 120

# 3. Label space \& mapping

In [3]:
CONDITIONS: List[str] = [
    "DEPRESSION",
    "ANXIETY",
    "SUICIDAL",
    "STRESS",
    "BIPOLAR",
    "PERSONALITY_DISORDER",
]

ALL_CLASSES: List[str] = ["NORMAL"] + CONDITIONS

SPECIAL_LABELS = {"OUT_OF_SCOPE", "INSUFFICIENT"}  # not expected in aspects-only prompt, but tolerated

ASPECT_STRENGTHS = {"none", "weak", "clear"}

# When the prompt does NOT emit scope, we still keep these df columns for backwards compatibility.
SCOPE_DEFAULTS = {
    "in_scope": True,
    "sufficient_info": True,
    "evidence_type": "unknown_not_collected",
    "cues": "SCOPE_NOT_COLLECTED",
}

# 4. mini-4o batch instructions (mental-health specific)

In [4]:
BATCH_ANNOTATION_INSTRUCTIONS_V2 = """
You are an expert annotator for mental health–related social media posts.
You are NOT a clinician and must NOT give medical or treatment advice.

Your task has TWO parts:
PART A) Assign EXACTLY ONE primary label (same as before).
PART B) Independently indicate whether EACH condition has supporting textual evidence (comorbidity allowed).

═══════════════════════════════════════════════════════════════════
PART A: SINGLE PRIMARY LABEL (EXACTLY ONE)
═══════════════════════════════════════════════════════════════════
You must assign EXACTLY ONE of the following labels:

- NORMAL:
  No clear indication of significant mental distress.
  Everyday experiences, mild ups and downs, jokes, generic statements.

- ANXIETY:
  Language dominated by worry, fear, panic, nervousness, overthinking,
  racing thoughts, or physiological anxiety symptoms (e.g., "heart racing").
  Focus is on anticipation or fear about future events.

- DEPRESSION:
  Persistent low mood, sadness, emptiness, hopelessness, loss of interest,
  low energy, guilt, feeling like a burden, but WITHOUT explicit suicidal intent.

- SUICIDAL:
  Explicit or strongly implied desire to die, self-harm, or kill oneself.
  If both depression and suicidal intent are present, choose SUICIDAL.

- STRESS:
  Overload or pressure tied to specific situations (work, exams, finances,
  family conflict) without pervasive hopelessness of depression.

- BIPOLAR:
  Mentions of bipolar disorder / manic episodes / clear bipolarity indicators.

- PERSONALITY_DISORDER:
  Mentions of PDs (e.g., borderline, narcissistic, antisocial) OR strong patterns
  suggesting PD traits (unstable relationships, abandonment fear, identity disturbance).

Borderline rules (same as before):
- If clearly suicidal ideation/intent → SUICIDAL.
- If "stressed" + strong hopelessness/pervasive low mood → DEPRESSION.
- If vague "mental health" w/o clear clues, neutral tone → NORMAL.
- If quoting/discussing generally without own experience, choose best-fit tone; if neutral → NORMAL.

═══════════════════════════════════════════════════════════════════
PART B: MULTI-ASPECT EVIDENCE FLAGS (COMORBIDITY ALLOWED)
═══════════════════════════════════════════════════════════════════
After choosing the single primary label, independently set evidence flags for each condition below.
Multiple conditions can be true simultaneously.

For each condition, output:
- present: true/false
- strength: one of {"none","weak","clear"}  (use conservative judgment)
- cues: 2–6 words capturing the key textual evidence (NOT a full explanation)

Conditions to flag:
- DEPRESSION
- ANXIETY
- SUICIDAL
- STRESS
- BIPOLAR
- PERSONALITY_DISORDER

Evidence guidelines:
- present=true ONLY if there is textual evidence in the post (self-report OR clearly attributed).
- "weak": one mild cue or indirect language
- "clear": explicit mention or multiple strong cues
- For SUICIDAL: present=true requires explicit or strongly implied ideation/intent (not metaphors).
  Metaphors like "this is killing me" → present=false.
- For BIPOLAR / PERSONALITY_DISORDER: be conservative; if not explicit, usually "weak" at most.

Important consistency rule:
- The primary label must still be exactly one label (Part A).
- The multi-aspect flags do NOT need to sum to one and do NOT change the primary label.

═══════════════════════════════════════════════════════════════════
INPUT / OUTPUT FORMAT
═══════════════════════════════════════════════════════════════════
Input: JSON array of objects:
  {"index": int, "text": string}

Output: JSON array of same length and order. Each element:

{
  "index": <same integer>,
  "label": "<one of NORMAL, ANXIETY, DEPRESSION, SUICIDAL, STRESS, BIPOLAR, PERSONALITY_DISORDER>",
  "probs": {
    "NORMAL": <float>, "ANXIETY": <float>, "DEPRESSION": <float>, "SUICIDAL": <float>,
    "STRESS": <float>, "BIPOLAR": <float>, "PERSONALITY_DISORDER": <float>
  },
  "aspects": {
    "DEPRESSION": {"present": <true/false>, "strength": "<none|weak|clear>", "cues": "<2-6 words>"},
    "ANXIETY": {"present": <true/false>, "strength": "<none|weak|clear>", "cues": "<2-6 words>"},
    "SUICIDAL": {"present": <true/false>, "strength": "<none|weak|clear>", "cues": "<2-6 words>"},
    "STRESS": {"present": <true/false>, "strength": "<none|weak|clear>", "cues": "<2-6 words>"},
    "BIPOLAR": {"present": <true/false>, "strength": "<none|weak|clear>", "cues": "<2-6 words>"},
    "PERSONALITY_DISORDER": {"present": <true/false>, "strength": "<none|weak|clear>", "cues": "<2-6 words>"}
  }
}

Rules:
- Output length must equal input length; preserve order by index.
- probs must be non-negative and sum to 1.0 (within rounding).
- aspects must contain exactly the 6 keys above.
- No extra fields. No explanations. No markdown. Must be valid JSON.
"""

# 5. Adapted batch annotation function

In [5]:
def _safe_float(x: Any, default: float = 0.0) -> float:
    try:
        v = float(x)
        if not np.isfinite(v):
            return default
        return v
    except Exception:
        return default


def _normalize_probs(probs_raw: Dict[str, Any], label_fallback: str) -> Dict[str, float]:
    """Clip per-condition probabilities to [0, 1]. No sum-to-1 constraint."""
    result = {}
    for c in ALL_CLASSES:
        v = _safe_float(probs_raw.get(c, 0.0), 0.0)
        result[c] = float(np.clip(v, 0.0, 1.0))
    return result


def _parse_aspects(aspects_raw: Any) -> Dict[str, Dict[str, Any]]:
    aspects: Dict[str, Dict[str, Any]] = {}
    src = aspects_raw if isinstance(aspects_raw, dict) else {}

    for cond in CONDITIONS:
        a = src.get(cond, {}) if isinstance(src.get(cond, {}), dict) else {}
        present = bool(a.get("present", False))
        strength = str(a.get("strength", "none")).strip().lower()
        if strength not in ASPECT_STRENGTHS:
            strength = "clear" if present else "none"
        cues = str(a.get("cues", ""))[:120]

        if not present:
            strength = "none"

        aspects[cond] = {"present": present, "strength": strength, "cues": cues}

    return aspects


def _parse_single_item(item: Dict[str, Any], model_version: str, system_fp: str, raw: str) -> Dict[str, Any]:
    label = str(item.get("label", "")).strip()
    if label not in set(ALL_CLASSES) | SPECIAL_LABELS:
        label = "NORMAL"

    probs_norm = _normalize_probs(item.get("probs", {}) or {}, label)
    parsed_aspects = _parse_aspects(item.get("aspects", {}) or {})
    scope = dict(SCOPE_DEFAULTS)

    return {
        "scope": scope,
        "aspects": parsed_aspects,
        "probs": probs_norm,
        "label": label,
        "model_version": model_version,
        "system_fingerprint": system_fp,
        "raw_json": raw,
    }


def _make_fallback_result(model_version: str, system_fp: str, raw: str) -> Dict[str, Any]:
    return {
        "scope": {
            "in_scope": False,
            "sufficient_info": False,
            "evidence_type": "unclear",
            "cues": "PARSE_FAILURE",
        },
        "aspects": {cond: {"present": False, "strength": "none", "cues": "PARSE_FAILURE"} for cond in CONDITIONS},
        "probs": {c: 0.0 for c in ALL_CLASSES},
        "label": "NORMAL",
        "model_version": model_version,
        "system_fingerprint": system_fp,
        "raw_json": raw,
    }


async def annotate_batch_async(
    texts: Sequence[str],
    aclient: AsyncOpenAI,
    semaphore: asyncio.Semaphore,
    model: str,
    prompt: str,
    max_retries: int,
    sleep_sec: float,
) -> Tuple[Optional[List[Dict[str, Any]]], int]:
    payload = [{"index": i, "text": t} for i, t in enumerate(texts)]
    payload_str = json.dumps(payload, ensure_ascii=False)

    async with semaphore:
        last_err: Any = None
        for attempt in range(max_retries):
            try:
                response = await aclient.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": prompt},
                        {"role": "user", "content": payload_str},
                    ],
                    temperature=0,
                )

                model_version = response.model
                system_fp = getattr(response, "system_fingerprint", "N/A")
                raw = response.choices[0].message.content.strip()

                if raw.startswith("```"):
                    raw = raw.strip("`")
                    raw = raw.replace("json", "", 1).strip()

                data = json.loads(raw)
                if not isinstance(data, list) or len(data) != len(texts):
                    raise ValueError(
                        f"Length mismatch: got {len(data) if isinstance(data, list) else 'non-list'}, expected {len(texts)}"
                    )

                data_sorted = sorted(data, key=lambda x: x.get("index", 0))

                results: List[Dict[str, Any]] = []
                parse_failures = 0
                for item in data_sorted:
                    try:
                        results.append(_parse_single_item(item, model_version, system_fp, raw))
                    except Exception:
                        parse_failures += 1
                        results.append(_make_fallback_result(model_version, system_fp, raw))

                return results, parse_failures

            except RateLimitError:
                wait = min(60, (2 ** attempt) * 5)
                await asyncio.sleep(wait)
                last_err = "RateLimitError"
            except Exception as e:
                last_err = e
                await asyncio.sleep(sleep_sec)

        return None, 0


def _init_output_columns(df: pd.DataFrame) -> pd.DataFrame:
    if "u_label" not in df.columns:
        df["u_label"] = None

    for cls in ALL_CLASSES:
        col = f"u_p_{cls.lower()}"
        if col not in df.columns:
            df[col] = None

    for col in ["u_in_scope", "u_sufficient_info", "u_evidence_type", "u_scope_cues"]:
        if col not in df.columns:
            df[col] = None

    for cond in CONDITIONS:
        cl = cond.lower()
        for suffix in ["_present", "_strength", "_cues"]:
            col = f"u_{cl}{suffix}"
            if col not in df.columns:
                df[col] = None

    for col in ["u_model_version", "u_system_fingerprint", "u_annotation_timestamp", "u_raw_json"]:
        if col not in df.columns:
            df[col] = None

    return df


def _write_results_to_df(df: pd.DataFrame, batch_idx: Sequence[int], results: Sequence[Dict[str, Any]]) -> None:
    timestamp = datetime.utcnow().isoformat()

    for row_idx, res in zip(batch_idx, results):
        df.at[row_idx, "u_in_scope"] = res["scope"]["in_scope"]
        df.at[row_idx, "u_sufficient_info"] = res["scope"]["sufficient_info"]
        df.at[row_idx, "u_evidence_type"] = res["scope"]["evidence_type"]
        df.at[row_idx, "u_scope_cues"] = res["scope"]["cues"]

        df.at[row_idx, "u_label"] = res["label"]
        for cls in ALL_CLASSES:
            df.at[row_idx, f"u_p_{cls.lower()}"] = res["probs"][cls]

        for cond in CONDITIONS:
            cl = cond.lower()
            a = res["aspects"][cond]
            df.at[row_idx, f"u_{cl}_present"] = bool(a["present"])
            df.at[row_idx, f"u_{cl}_strength"] = a["strength"]
            df.at[row_idx, f"u_{cl}_cues"] = a["cues"]

        df.at[row_idx, "u_model_version"] = res["model_version"]
        df.at[row_idx, "u_system_fingerprint"] = res["system_fingerprint"]
        df.at[row_idx, "u_annotation_timestamp"] = timestamp
        df.at[row_idx, "u_raw_json"] = res["raw_json"]


async def run_annotation_async(
    df: pd.DataFrame,
    text_col: str,
    output_path: str,
    prompt: str,
    cfg: RuntimeConfig,
) -> pd.DataFrame:
    df = _init_output_columns(df)

    rows_to_do = df[df["u_label"].isna()].index.tolist()
    print(f"Rows to annotate: {len(rows_to_do)} / {len(df)}")

    if not rows_to_do:
        print("All rows already annotated.")
        return df

    aclient = AsyncOpenAI()
    semaphore = asyncio.Semaphore(cfg.max_concurrent)

    batches: List[Tuple[List[int], List[str]]] = []
    for start in range(0, len(rows_to_do), cfg.batch_size):
        batch_idx = rows_to_do[start : start + cfg.batch_size]
        texts = df.loc[batch_idx, text_col].fillna("").astype(str).tolist()
        batches.append((batch_idx, texts))

    n_batches = len(batches)
    print(f"Total batches: {n_batches} | Concurrency: {cfg.max_concurrent}")

    stats = {
        "total_batches": n_batches,
        "successful_batches": 0,
        "failed_batches": 0,
        "parse_fallbacks": 0,
    }

    wave_size = cfg.max_concurrent * 2
    last_checkpoint = time.time()
    pbar = tqdm(total=n_batches, desc="Annotating")

    for wave_start in range(0, n_batches, wave_size):
        wave = batches[wave_start : wave_start + wave_size]

        tasks = [
            annotate_batch_async(
                texts=texts,
                aclient=aclient,
                semaphore=semaphore,
                model=cfg.model,
                prompt=prompt,
                max_retries=cfg.max_retries,
                sleep_sec=cfg.sleep_sec,
            )
            for _, texts in wave
        ]

        results_list = await asyncio.gather(*tasks)

        for (batch_idx, _), (results, n_fallbacks) in zip(wave, results_list):
            if results is not None:
                _write_results_to_df(df, batch_idx, results)
                stats["successful_batches"] += 1
                stats["parse_fallbacks"] += int(n_fallbacks)
            else:
                stats["failed_batches"] += 1
            pbar.update(1)

        if time.time() - last_checkpoint > cfg.checkpoint_every_sec:
            df.to_csv(output_path, index=False)
            elapsed = time.time() - last_checkpoint
            print(f"\n  Checkpoint saved ({stats['successful_batches']}/{n_batches} batches, {elapsed:.0f}s)")
            last_checkpoint = time.time()

    pbar.close()

    df.to_csv(output_path, index=False)
    print(f"\nAnnotation complete. Stats: {json.dumps(stats, indent=2)}")

    return df


def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    prob_cols = [f"u_p_{c.lower()}" for c in ALL_CLASSES]
    for col in prob_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).clip(0.0, 1.0)

    eval_mask = (df["u_in_scope"] == True) & (df["u_sufficient_info"] == True)

    present_cols = [f"u_{c.lower()}_present" for c in CONDITIONS]
    present_matrix = df[present_cols].fillna(False).astype(bool).values

    df["u_n_conditions"] = 0
    df.loc[eval_mask, "u_n_conditions"] = present_matrix[eval_mask.values].sum(axis=1)

    df["u_any_clinical"] = eval_mask & (df["u_n_conditions"] > 0)
    df["u_normal_inferred"] = eval_mask & (df["u_n_conditions"] == 0)

    multilabel = []
    for i in range(len(df)):
        if not eval_mask.iloc[i]:
            multilabel.append("OUT_OF_SCOPE")
            continue
        active = [CONDITIONS[j] for j in range(len(CONDITIONS)) if present_matrix[i, j]]
        multilabel.append("+".join(active) if active else "NORMAL")
    df["u_multilabel"] = multilabel

    clin_prob_cols = [f"u_p_{c.lower()}" for c in CONDITIONS]
    df["u_max_clinical_prob"] = df[clin_prob_cols].max(axis=1)
    df["u_sum_clinical_prob"] = df[clin_prob_cols].sum(axis=1)

    return df


def print_summary(df: pd.DataFrame) -> None:
    n = len(df)
    print(f"\n{'=' * 60}\nSCREENING ANNOTATION SUMMARY\n{'=' * 60}")
    print(f"Total posts: {n}")

    n_in_scope = int(df["u_in_scope"].fillna(False).sum())
    n_oos = n - n_in_scope
    print(f"\nScope:")
    print(f"  In-scope:       {n_in_scope:6d} ({n_in_scope/max(n,1)*100:.1f}%)")
    print(f"  Out-of-scope:   {n_oos:6d} ({n_oos/max(n,1)*100:.1f}%)")

    print(f"\nPer-condition prevalence (aspect present=True):")
    for cond in CONDITIONS:
        col = f"u_{cond.lower()}_present"
        if col in df.columns:
            pos = int(df[col].fillna(False).sum())
            print(f"  {cond:25s}  {pos:>6d} ({pos/max(n,1)*100:.1f}%)")

    if "u_n_conditions" in df.columns:
        print(f"\nComorbidity distribution (in-scope):")
        mask = df["u_in_scope"] == True
        print(df.loc[mask, "u_n_conditions"].value_counts().sort_index().to_string())

    print(f"\nSoft probability stats (in-scope):")
    mask = df["u_in_scope"] == True
    for cond in CONDITIONS:
        col = f"u_p_{cond.lower()}"
        if col in df.columns:
            vals = df.loc[mask, col]
            print(f"  {cond:25s}  mean={vals.mean():.3f}  >0.5={int((vals>0.5).sum()):>5d}  >0.0={int((vals>0).sum()):>5d}")


def write_clean_csv(input_path: str, output_path: str, drop_cols: Optional[List[str]] = None) -> None:
    drop_cols = drop_cols or []
    df = pd.read_csv(input_path)
    for c in drop_cols:
        if c in df.columns:
            df = df.drop(columns=[c])

    obj_cols = df.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        df[c] = df[c].astype(str).str.replace("\n", " ").str.replace("\r", " ")

    df.to_csv(output_path, index=False)

## 6. Execution

In [6]:
BASE_DIR = "/content/drive/MyDrive/NLP_Projects/02_MentalHealth/00_Labelling/data"
input_path = os.path.join(BASE_DIR, "Combined Data.csv")
output_path = os.path.join(BASE_DIR, "mental_health_unified_labels_clean.csv")

cfg = RuntimeConfig(
    model="gpt-4o-mini",
    batch_size=20,
    max_concurrent=15,
    max_retries=3,
    sleep_sec=2.0,
    checkpoint_every_sec=120,
)

df = pd.read_csv(input_path)
print(f"Loaded {len(df)} rows")

df = await run_annotation_async(
    df=df,
    text_col="statement",
    output_path=output_path,
    prompt=BATCH_ANNOTATION_INSTRUCTIONS_V2,
    cfg=cfg,
)

df = add_derived_columns(df)
df.to_csv(output_path, index=False)
print_summary(df)

Loaded 53043 rows
Rows to annotate: 53043 / 53043
Total batches: 2653 | Concurrency: 15


Annotating:   0%|          | 0/2653 [00:00<?, ?it/s]

/tmp/ipython-input-4904/2475182201.py:168: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()



  Checkpoint saved (30/2653 batches, 215s)

  Checkpoint saved (60/2653 batches, 294s)

  Checkpoint saved (90/2653 batches, 196s)

  Checkpoint saved (120/2653 batches, 190s)

  Checkpoint saved (150/2653 batches, 211s)

  Checkpoint saved (180/2653 batches, 224s)

  Checkpoint saved (210/2653 batches, 209s)

  Checkpoint saved (240/2653 batches, 223s)

  Checkpoint saved (270/2653 batches, 234s)

  Checkpoint saved (300/2653 batches, 248s)

  Checkpoint saved (330/2653 batches, 216s)

  Checkpoint saved (360/2653 batches, 259s)

  Checkpoint saved (390/2653 batches, 216s)

  Checkpoint saved (420/2653 batches, 266s)

  Checkpoint saved (450/2653 batches, 518s)

  Checkpoint saved (480/2653 batches, 256s)

  Checkpoint saved (510/2653 batches, 303s)

  Checkpoint saved (540/2653 batches, 215s)

  Checkpoint saved (570/2653 batches, 293s)

  Checkpoint saved (600/2653 batches, 317s)

  Checkpoint saved (630/2653 batches, 314s)

  Checkpoint saved (660/2653 batches, 405s)

  Checkpoint

/tmp/ipython-input-4904/2475182201.py:281: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  present_matrix = df[present_cols].fillna(False).astype(bool).values



SCREENING ANNOTATION SUMMARY
Total posts: 53043

Scope:
  In-scope:        53023 (100.0%)
  Out-of-scope:       20 (0.0%)

Per-condition prevalence (aspect present=True):
  DEPRESSION                  20581 (38.8%)
  ANXIETY                     11214 (21.1%)
  SUICIDAL                    13201 (24.9%)
  STRESS                      10174 (19.2%)
  BIPOLAR                      1853 (3.5%)
  PERSONALITY_DISORDER         1244 (2.3%)

Comorbidity distribution (in-scope):
u_n_conditions
0    17463
1    17885
2    13504
3     3383
4      722
5       59
6        7

Soft probability stats (in-scope):
  DEPRESSION                 mean=0.182  >0.5= 9573  >0.0=16026
  ANXIETY                    mean=0.125  >0.5= 7347  >0.0= 9776
  SUICIDAL                   mean=0.212  >0.5=11567  >0.0=14012
  STRESS                     mean=0.093  >0.5= 2152  >0.0=19733
  BIPOLAR                    mean=0.023  >0.5= 1203  >0.0= 1519
  PERSONALITY_DISORDER       mean=0.016  >0.5=  389  >0.0= 3997


/tmp/ipython-input-4904/2475182201.py:310: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  n_in_scope = int(df["u_in_scope"].fillna(False).sum())
/tmp/ipython-input-4904/2475182201.py:320: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pos = int(df[col].fillna(False).sum())


# 5. Check labelling quality and clean

In [8]:
import csv

BASE_DIR = "/content/drive/MyDrive/NLP_Projects/02_MentalHealth/00_Labelling/data"
input_path = os.path.join(BASE_DIR, "mental_health_unified_labels_clean.csv")
output_path = os.path.join(BASE_DIR, "mental_health_unified_labels_final.csv")

with open(input_path) as fin:
    reader = csv.DictReader(fin)
    fieldnames = [f for f in reader.fieldnames if f != "u_raw_json"]

    with open(output_path, "w", newline="") as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()

        for r in reader:
            del r["u_raw_json"]
            for k in r:
                r[k] = r[k].replace("\n", " ").replace("\r", "")
            writer.writerow(r)

print(f"Done. Wrote {output_path}")


Done. Wrote /content/drive/MyDrive/NLP_Projects/02_MentalHealth/00_Labelling/data/mental_health_unified_labels_final.csv


In [9]:
BASE_DIR = "/content/drive/MyDrive/NLP_Projects/02_MentalHealth/00_Labelling/data"
input_path = os.path.join(BASE_DIR, "mental_health_unified_labels_clean.csv")
output_path = os.path.join(BASE_DIR, "mental_health_unified_labels_final.csv")

cfg = RuntimeConfig(
    model="gpt-4o-mini",
    batch_size=10,
    max_concurrent=15,
    max_retries=3,
    sleep_sec=2.0,
    checkpoint_every_sec=120,
)

df = pd.read_csv(input_path)
print(f"Loaded {len(df)} rows")

df = await run_annotation_async(
    df=df,
    text_col="statement",
    output_path=output_path,
    prompt=BATCH_ANNOTATION_INSTRUCTIONS_V2,
    cfg=cfg,
)

df = add_derived_columns(df)
df.to_csv(output_path, index=False)
print_summary(df)

Loaded 53043 rows
Rows to annotate: 20 / 53043
Total batches: 2 | Concurrency: 15


/tmp/ipython-input-4904/1169385529.py:14: DtypeWarning: Columns (11,12,15,18,21,24,27,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


Annotating:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipython-input-4904/2475182201.py:168: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()



Annotation complete. Stats: {
  "total_batches": 2,
  "successful_batches": 2,
  "failed_batches": 0,
  "parse_fallbacks": 0
}


/tmp/ipython-input-4904/2475182201.py:281: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  present_matrix = df[present_cols].fillna(False).astype(bool).values



SCREENING ANNOTATION SUMMARY
Total posts: 53043

Scope:
  In-scope:        53043 (100.0%)
  Out-of-scope:        0 (0.0%)

Per-condition prevalence (aspect present=True):
  DEPRESSION                  20597 (38.8%)
  ANXIETY                     11217 (21.1%)
  SUICIDAL                    13211 (24.9%)
  STRESS                      10179 (19.2%)
  BIPOLAR                      1853 (3.5%)
  PERSONALITY_DISORDER         1245 (2.3%)

Comorbidity distribution (in-scope):
u_n_conditions
0    17463
1    17893
2    13514
3     3384
4      723
5       59
6        7

Soft probability stats (in-scope):
  DEPRESSION                 mean=0.183  >0.5= 9584  >0.0=16042
  ANXIETY                    mean=0.125  >0.5= 7348  >0.0= 9777
  SUICIDAL                   mean=0.212  >0.5=11572  >0.0=14024
  STRESS                     mean=0.093  >0.5= 2152  >0.0=19742
  BIPOLAR                    mean=0.023  >0.5= 1203  >0.0= 1519
  PERSONALITY_DISORDER       mean=0.016  >0.5=  389  >0.0= 3997


/tmp/ipython-input-4904/2475182201.py:310: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  n_in_scope = int(df["u_in_scope"].fillna(False).sum())
/tmp/ipython-input-4904/2475182201.py:320: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pos = int(df[col].fillna(False).sum())
